# The Maths of Data Science and AI - Introduction to text analysis

## Introduction
### Data context

This is a list of 1,000 hotels and their reviews crawled from booking.com. The dataset includes hotel name and rating, review rating, a qualitative review and title.

						

### Data features

| Feature       | Description     | Notes |
| ---           | ---             | ---   |
|**review_title**  | Title of review |       |
|**reviewed_at**	| Date of review  |       |
|**hotel_name**	    | Name of hotel reviewed  |       |
|**avg_rating**     | Average rating of hotel reviewed  |    |
|**rating**	        | Individual review rating |   1:bad - 5:good|
|**review_text**    | |    |



## Importing libraries and data

### Importing the libraries

In [ ]:
import pandas as pd
import seaborn as sns
import re

#To display the full sentences
pd.set_option('display.max_colwidth', None)

# There are currently some unhelpful warnings
import warnings
warnings.filterwarnings('ignore')

# required for Bag of Words
from collections import Counter
from nltk.util import ngrams

# To create an Natural Language Processing classification model
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Function to draw the model
def plot_decision_tree(tree_model):
    fig, ax = plt.subplots(figsize=(20,8))
    plot_tree(tree_model,  
        filled=True, 
        impurity=False, 
        feature_names= vectorizer.get_feature_names_out(), 
        class_names=["No","Yes"], 
        proportion=True, 
        ax=ax)
    plt.show()

### Importing the dataset

In [ ]:
# import the dataset and use helpful feature names
review_data = pd.read_csv('/kaggle/input/hotel-reviews/Datafiniti_Hotel_Reviews_Jun19.csv', usecols=['reviews.title', 'reviews.date', 'name', 'reviews.rating', 'reviews.text'])
reviews_data = review_data.rename(columns={"reviews.title": "review_title", "reviews.date": "reviewed_at", "reviews.rating": "rating", "reviews.text": "review_text"})
reviews_data

In [ ]:
# explore the dataset
reviews_data.info()

# Preparing the Data
This task will only use the reviews.rating and reviews.text columns. The punctuation can be removed to aid analysis.

In [ ]:
# Use re to prep the text - remove punctuation or capitals
reviews_data['review_text'] = reviews_data['review_text'].str.replace(r'[^\w\s]',' ', regex=True).str.lower()

# Replace numerical scores with a categorical description
reviews_data['sentiment'] = reviews_data['rating'].replace({
    1: "Terrible",
    2: "Poor",
    3: "Neutral",
    4: "Good",
    5: "Excellent"})

# find the number of each review rating
reviews_data['rating'].value_counts()

In [ ]:
# display the distribution of review ratings
sns.displot(reviews_data, x='rating', binwidth=0.5, binrange=(0.75,5.25), aspect=2);

For this task reviews will be classified with the binary feature `positive_review`: 1 (positive) or 0 (not positive). 

In [ ]:
# create a binary feature (1 or 0) and classify reviews with a rating of more than 3 as a positive review
reviews_data['positive_review'] = (reviews_data['rating']>3).astype(int)

# display the number of each
reviews_data['positive_review'].value_counts()

# Tokenization

Tokenization splits each piece of text into individual *tokens* to be analysed. An example is given below for the first review.

In [ ]:
# get an example of a review and display it
sample_review = reviews_data['review_text'][0]
sample_review

## Term frequency
The *term frequency* for each word in a review can be found. This uses a *bag of words* approach, i.e. the order of the words is not taken into account.

In [ ]:
# Find the term frequency of each token using the Counter function
tokens_one = sample_review.split()
counts = Counter(tokens_one)
counts

## Term Frequency-Inverse Document Frequency (TF-IDF)
The TF-IDF score for each word gives a value that indicates the importance the word by comparing its frequency in individual reviews compared to the full set of reviews. The code block below calculates this for all the words in the data set.

In [ ]:
#create the IDF value for the words in each review
all_reviews = reviews_data['review_text']

# Step 1: Term frequencies (Bag of words for each review)
vectorizer = CountVectorizer()
term_frequencies = vectorizer.fit_transform(all_reviews)

# Step 2: TF-IDF transformation
transformer = TfidfTransformer()
tfidf_matrix = transformer.fit_transform(term_frequencies)

Inverse Document Frequency values can be found for individual words. *"noisy"* is a more important words than *"the"*.

In [ ]:
# Calculate IDF values and create a table
idf_values = transformer.idf_
words = vectorizer.get_feature_names_out()
idf_values = pd.DataFrame({'word': words, 'idf': idf_values})

# display idf values for selected words
idf_values[(idf_values['word']=='the')|(idf_values['word']=='noisy')|(idf_values['word']=='clean')]

# Task: Create some decision tree models and try some sentences
The code below builds a model of depth 2. The model above will score a sentence as **Yes** (positive) or **No** (not positive).

* Try your own sentences ​
* Can you predict a sentence using the decision tree that will give a
positive or negative review?​
* Explore how changing the depth of the model affects the predictions

In [ ]:
# define the input and output variables 
X = reviews_data['review_text']
y = reviews_data['positive_review']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=1)

# prepare the data by converting to vectors of TF-IDF scores
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# create the model with the training data and display
tree_model = DecisionTreeClassifier(max_depth=2, random_state=1)
tree_model.fit(X_train_tfidf, y_train)
plot_decision_tree(tree_model)

# create some predictions with the testing data and display the metrics
y_pred = tree_model.predict(X_test_tfidf)
print(pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted'], margins=True))
print("Precision: ",round(100*precision_score(y_test, y_pred),1),"%")
print("Recall: ",round(100*recall_score(y_test, y_pred),1),"%")

## Trying some sentences

In [ ]:
# sample sentence
sample_sentence = ["The rooms were clean and the service was great"]

# create and display prediction
prediction = tree_model.predict(vectorizer.transform(sample_sentence))
print(f"'{sample_sentence}' → {prediction}")

In [ ]:
# sample sentence
sample_sentence = ["The room was really comfortable"]

# create and display prediction
prediction = tree_model.predict(vectorizer.transform(sample_sentence))
print(f"'{sample_sentence}' → {prediction}")

In [ ]:
# sample sentence
sample_sentence = ["The location was not great, the room was not clean and the staff were not friendly. sorry"]

# create and display prediction
prediction = tree_model.predict(vectorizer.transform(sample_sentence))
print(f"'{sample_sentence}' → {prediction}")

* Try with some sentences in the code block below.
* Use the tree to create your own sentence that you think the model will classify incorrectly.

In [ ]:
# sample sentence


# create and display prediction
